# Test Retrieval from Milvus and SQLite Databases

This notebook tests:
1. Vector similarity search from Milvus Lite database
2. Reranking results using a cross-encoder
3. Querying structured data from SQLite3 database
4. Combining results from both databases

## 1. Setup and Import Dependencies

In [ ]:
import sqlite3
from langchain_milvus import Milvus
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

print("All dependencies imported successfully!")

## 2. Load Milvus Vector Store and Setup Reranker

In [ ]:
# Initialize embeddings (same as used during data generation)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load existing Milvus vector store
URI = "../../data/parking.db"
vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": URI},
    collection_name="parking_policy_collection",
    drop_old=False  # Load existing data
)

print(f" Milvus vector store loaded from {URI}")

In [ ]:
# Setup reranker pipeline
print("Setting up reranker...")

# Base retriever fetches top-k candidates
base_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# Cross-encoder reranker model
reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

# Compressor that reranks and keeps top-n
compressor = CrossEncoderReranker(model=reranker_model, top_n=3)

# Final retrieval pipeline with reranking
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print(" Reranker setup complete!")

## 3. Connect to SQLite Database

In [ ]:
# Connect to SQLite database
sqlite_db_path = "../../data/parking_db.sqlite"
conn = sqlite3.connect(sqlite_db_path)
cursor = conn.cursor()

print(f" Connected to SQLite database: {sqlite_db_path}")

# Verify tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"\nAvailable tables: {[table[0] for table in tables]}")

## 4. Test Vector Retrieval with Reranking

In [ ]:
# Test 1: Basic similarity search (without reranking)
test_query_1 = "What are the operating hours?"

print(f"\n{'='*60}")
print(f"TEST 1: Basic Similarity Search")
print(f"Query: '{test_query_1}'")
print(f"{'='*60}\n")

basic_results = vector_store.similarity_search(test_query_1, k=3)

for i, doc in enumerate(basic_results, 1):
    print(f"--- Result {i} (Basic) ---")
    print(doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content)
    print()

In [ ]:
# Test 2: Retrieval with Reranking
test_query_2 = "Are heavy commercial trucks allowed?"

print(f"\n{'='*60}")
print(f"TEST 2: Retrieval with Reranking")
print(f"Query: '{test_query_2}'")
print(f"{'='*60}\n")

reranked_results = compression_retriever.invoke(test_query_2)

for i, doc in enumerate(reranked_results, 1):
    print(f"--- Reranked Result {i} ---")
    print(doc.page_content)
    print()

In [ ]:
# Test 3: Query about pricing
test_query_3 = "How much does parking cost per hour?"

print(f"\n{'='*60}")
print(f"TEST 3: Pricing Query with Reranking")
print(f"Query: '{test_query_3}'")
print(f"{'='*60}\n")

reranked_results = compression_retriever.invoke(test_query_3)

for i, doc in enumerate(reranked_results, 1):
    print(f"--- Reranked Result {i} ---")
    print(doc.page_content)
    print()

## 5. Test SQLite Database Queries

In [ ]:
# Test 4: Query available parking spots
print(f"\n{'='*60}")
print(f"TEST 4: Available Parking Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, spot_type, floor, status, price_per_hour
    FROM parking_spots
    WHERE status = 'available'
""")

available_spots = cursor.fetchall()

print(f"Found {len(available_spots)} available spots:\n")
for spot in available_spots:
    print(f"  • {spot[0]} - {spot[1]} ({spot[2]}) - ${spot[4]}/hour")

In [ ]:
# Test 5: Query EV charging spots
print(f"\n{'='*60}")
print(f"TEST 5: EV Charging Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, floor, status
    FROM parking_spots
    WHERE spot_type = 'EV'
""")

ev_spots = cursor.fetchall()

print(f"Found {len(ev_spots)} EV charging spots:\n")
for spot in ev_spots:
    print(f"  • {spot[0]} on {spot[1]} - Status: {spot[2]}")

In [ ]:
# Test 6: Query accessible spots
print(f"\n{'='*60}")
print(f"TEST 6: Accessible Parking Spots")
print(f"{'='*60}\n")

cursor.execute("""
    SELECT spot_number, floor, status
    FROM parking_spots
    WHERE spot_type = 'Accessible'
""")

accessible_spots = cursor.fetchall()

print(f"Found {len(accessible_spots)} accessible spots:\n")
for spot in accessible_spots:
    print(f"  • {spot[0]} on {spot[1]} - Status: {spot[2]}")

## 6. Combined Test: Policy + Database Query

In [ ]:
# Test 7: Combined retrieval - user asks about EV charging
user_query = "Where can I find EV charging stations and are they available?"

print(f"\n{'='*60}")
print(f"TEST 7: Combined Query (Vector + SQL)")
print(f"User Query: '{user_query}'")
print(f"{'='*60}\n")

# Step 1: Get policy information from vector store
print("[1] Retrieving policy information from vector store...\n")
policy_docs = compression_retriever.invoke(user_query)

print("Policy Information:")
for i, doc in enumerate(policy_docs, 1):
    print(f"\n--- Policy Excerpt {i} ---")
    print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)

# Step 2: Get real-time availability from SQLite
print("\n" + "="*60)
print("[2] Checking real-time EV spot availability in database...\n")

cursor.execute("""
    SELECT spot_number, floor, status, price_per_hour
    FROM parking_spots
    WHERE spot_type = 'EV'
""")

ev_spots = cursor.fetchall()

print("Real-time EV Spot Status:")
for spot in ev_spots:
    status_emoji = "" if spot[2] == 'available' else ""
    print(f"  {status_emoji} {spot[0]} on {spot[1]} - {spot[2].upper()} - ${spot[3]}/hour")

print("\n" + "="*60)
print("Combined Answer: Based on policy and real-time data")
print("="*60)

## 7. Database Statistics

In [ ]:
# Test 8: Overall statistics
print(f"\n{'='*60}")
print(f"DATABASE STATISTICS")
print(f"{'='*60}\n")

# Total spots
cursor.execute("SELECT COUNT(*) FROM parking_spots")
total_spots = cursor.fetchone()[0]
print(f"Total parking spots: {total_spots}")

# Available spots
cursor.execute("SELECT COUNT(*) FROM parking_spots WHERE status = 'available'")
available_count = cursor.fetchone()[0]
print(f"Available spots: {available_count}")

# By type
cursor.execute("""
    SELECT spot_type, COUNT(*), 
           SUM(CASE WHEN status = 'available' THEN 1 ELSE 0 END) as available
    FROM parking_spots
    GROUP BY spot_type
""")

type_stats = cursor.fetchall()
print("\nBreakdown by type:")
for stat in type_stats:
    print(f"  • {stat[0]}: {stat[2]}/{stat[1]} available")

# By floor
cursor.execute("""
    SELECT floor, COUNT(*),
           SUM(CASE WHEN status = 'available' THEN 1 ELSE 0 END) as available
    FROM parking_spots
    GROUP BY floor
""")

floor_stats = cursor.fetchall()
print("\nBreakdown by floor:")
for stat in floor_stats:
    print(f"  • {stat[0]}: {stat[2]}/{stat[1]} available")

## 8. Cleanup

In [ ]:
# Close SQLite connection
conn.close()
print("\n SQLite connection closed.")
print("\n" + "="*60)
print("All tests completed successfully!")
print("="*60)